# CIs for FINN via DDB: FINN, PI3NN, DDB and CI Construction

- FINN for PI3NN mean predictor on core 2 data
- PI3NN on core 2 data
- draw bootstraps
- FINN on bootstraps for CI

In [ ]:
from pathlib import Path
import importlib

import numpy as np
import matplotlib.pyplot as plt
import time

import finn.io as fio
import finn.engine as feng
import pi3nn.io as pio
import pi3nn.engine as peng
import ddb.io as dio
import ddb.engine as deng
import samples

importlib.reload(fio)
importlib.reload(feng)
importlib.reload(pio)
importlib.reload(peng)
importlib.reload(dio)
importlib.reload(deng)
importlib.reload(samples)

from samples import Samples

Setup directories

In [ ]:
input_dir = Path("../in")
output_dir = Path("../out")
results_dir = output_dir / "results"

finn_results_dir = fio.FINNDir(results_dir / "finn")
pi3nn_results_dir = pio.PI3NNDir(results_dir / "pi3nn")
ddb_results_dir = dio.DDBDir(results_dir / "ddb")

Load data

In [ ]:
core2 = fio.load_exp_data(f"Core 2")
core2b = fio.load_exp_data(f"Core 2B")
core1 = fio.load_exp_data(f"Core 1")

core2_cfg = fio.load_exp_cfg(name="Core 2")

print(core1.shape)
print(type(core1))

FINN

In [ ]:
dt = time.time()
feng.setup_and_train_model(finn_results_dir, core2, **core2_cfg)
dt = time.time() - dt
if not (results_dir / "finn_training_time.txt").exists():
    with open(results_dir / "finn_training_time.txt", "w") as file:
        file.write(f"{dt:.2f} s")
    print(f"FINN training time: {dt:.2f} s.")

In [ ]:
ret = finn_results_dir.best_ret
best_pred_c_btc = finn_results_dir.best_pred_c_btc
c_train = np.load(finn_results_dir.c_train_path)

In [ ]:
plt.figure(figsize=[10, 6])
plt.plot(c_train[:, 0], c_train[:, 1], 'x', label='Data')
plt.plot(c_train[:, 0], best_pred_c_btc, '-', label='FINN Prediction')

PI3NN on core 2 data with FINN

In [ ]:
quantiles = np.linspace(0.0, 1.0, 51, endpoint=True)
print(quantiles)

In [ ]:
dt = time.time()
peng.compute_quantiles(pi3nn_results_dir, finn_results_dir, quantiles)
dt = time.time() - dt
if not (results_dir / "quantile_computing_time.txt").exists():
    with open(results_dir / "quantile_computing_time.txt", "w") as file:
        file.write(f"{dt:.2f} s")
    print(f"Quantile computing time: {dt:.2f} s.")

In [ ]:
upper_res = np.load(pi3nn_results_dir.c_pred_upper_res_path)
lower_res = np.load(pi3nn_results_dir.c_pred_lower_res_path)
median = np.load(pi3nn_results_dir.c_pred_median_path)

t_upper_res_train = np.load(pi3nn_results_dir.upper_dir / "t_train.npy")
t_lower_res_train = np.load(pi3nn_results_dir.lower_dir / "t_train.npy")
upper_res_train = np.load(pi3nn_results_dir.upper_dir / "c_res_train.npy")
lower_res_train = np.load(pi3nn_results_dir.lower_dir / "c_res_train.npy")
upper_res_pred_train = np.load(pi3nn_results_dir.upper_dir / "c_res_pred_train.npy")
lower_res_train_res_pred_train = np.load(pi3nn_results_dir.lower_dir / "c_res_pred_train.npy")

plt.plot(t_upper_res_train, upper_res_train, 'rx')
plt.plot(t_upper_res_train, upper_res_pred_train, 'r-')
plt.figure()
plt.plot(t_lower_res_train, lower_res_train, 'bx')
plt.plot(t_lower_res_train, lower_res_train_res_pred_train, 'b-')
plt.figure(figsize=[10, 6])
plt.plot(c_train[:, 0], c_train[:, 1], 'x', label='Data')
plt.plot(c_train[:, 0], median, '-', label='Median')
plt.plot(c_train[:, 0], median + upper_res, 'r--', label='Upper Res')
plt.plot(c_train[:, 0], median - lower_res, 'b--', label='Lower Res')

In [ ]:
quantiles_data = pi3nn_results_dir.iter_quantiles()

plt.figure(figsize=[10, 6])
for quantile in quantiles_data.keys():
    q_data = quantiles_data[quantile]

    plt.plot(c_train[:, 0], q_data, ".")

DDB on computed quantiles

In [ ]:
dt = time.time()
deng.setup_and_train_ddb(ddb_results_dir, pi3nn_results_dir, **core2_cfg)
dt = time.time() - dt
if not (results_dir / "ddb_finns_training_time.txt").exists():
    with open(results_dir / "ddb_finns_training_time.txt", "w") as file:
        file.write(f"{dt:.2f} s")
    print(f"DDB FINNs training time: {dt:.2f} s.")

In [ ]:
ddb_samples = Samples.from_dir2(ddb_results_dir.path)
c_train = np.load(ddb_results_dir.c_train_path)

In [ ]:
plt.figure(figsize=[10, 6])
plt.plot(c_train[:, 0], ddb_samples.mixed_quantiles, '.'
         )

In [ ]:
plt.plot(c_train[:, 0], c_train[:, 1], "x", markersize=1)
plt.plot(c_train[:, 0], ddb_samples.core2[:, :], "-")
plt.plot(c_train[:, 0], ddb_samples.mixed_quantiles[:, 0], ".")

plt.figure()
plt.plot(core1["time"], core1["c_diss"], "x", markersize=1)
plt.plot(core1["time"], ddb_samples.core1[:, :], "-")

plt.figure()
plt.plot(core2b["x"], core2b["c_tot"], "x", markersize=1)
plt.plot(core2b["x"], ddb_samples.core2b[:, :], "-")

plt.figure()
plt.plot(ddb_samples.ret_x, ddb_samples.ret_y, "-")